# Practice Session 04: Networks from text

Author: <font color="blue">Martí Pascual Barluenga</font>

E-mail: <font color="blue">marti.pascual01@estudiant.upf.edu</font>

Date: <font color="blue">15/10/2024</font>

# 1. Create the directed mention network

In [2]:
import io
import json
import gzip
import csv
import re

from IPython.display import Image

In [3]:
# Leave this code as-is

# Input file
COMPRESSED_INPUT_FILENAME = "CovidLockdownCatalonia.json.gz"

# These are the output files, leave as-is
OUTPUT_ALL_EDGES_FILENAME = "CovidLockdownCatalonia.csv"
OUTPUT_FILTERED_EDGES_FILENAME = "CovidLockdownCatalonia-min-weight-filtered.csv"
OUTPUT_CO_MENTIONS_FILENAME = "CovidLockdownCatalonia-co-mentions.csv"

## 1.1. Extract mentions

In [4]:
# Leave this code as-is

def extract_mentions(text):
    return re.findall("@([a-zA-Z0-9_]{5,20})", text)

print(extract_mentions("RT @elnacionalcat: check this post by @en_blau"))

['elnacionalcat', 'en_blau']


## 1.2. Count mentions


In [5]:
mentions_counter = {}

with gzip.open(COMPRESSED_INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
    for line in input_file:
        tweet = json.loads(line)
        author = tweet["user"]["screen_name"]
        message = tweet["full_text"]
        mentions = extract_mentions(message)
        for mention in mentions:
            key = (author, mention)
            if key in mentions_counter:
                mentions_counter[key] += 1
            else:
                mentions_counter[key] = 1

In [6]:
print( mentions_counter[('joanmariapique', 'catalangov')])

9


In [7]:
mutual_mentions_counter = {}

for (author, mention), count in mentions_counter.items():
    if (mention, author) in mentions_counter:
        mutual_count = count + mentions_counter[(mention, author)]
        if mutual_count > 8:

            if (mention, author) not in mutual_mentions_counter:
                mutual_mentions_counter[(author, mention)] = mutual_count

for (author, mention), mutual_count in mutual_mentions_counter.items():
    print(f"Accounts @{author} and @{mention} mention each other {mutual_count} times")

Accounts @MargaXrepublica and @MargaXrepublica mention each other 12 times


In [8]:
# Leave this code as-is

with io.open(OUTPUT_ALL_EDGES_FILENAME, "w") as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"')
    writer.writerow(["Source", "Target", "Weight"])
    for key in mentions_counter:
        author = key[0]
        mention = key[1]
        weight = mentions_counter[key]
        writer.writerow([author, mention, weight])

In [9]:
with io.open(OUTPUT_FILTERED_EDGES_FILENAME, "w") as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"')
    writer.writerow(["Source", "Target", "Weight"])
    for key, weight in mentions_counter.items():
        if weight >= 3:
            author, mention = key
            writer.writerow([author, mention, weight])

## 2.1. Visualize the largest connected component


Style nodes in the largest connected component:

* Run `Tools > Analyze Network ...` -- select **directed graph** because this graph is directed
* Node size and label size proportional to their in-degree
* Node color proportional to in-degree (white=small, blue=large)
* Edge width proportional to weight

Save the image as `mentions-largest-cc.png`, the next cell should display it. It is OK if your image does not look exactly like the example we provide.

*Tip*: The file is large so if you want to see all details while zooming out you may have to set ``View > Always show Graphic Details``. Note this makes the program run slower.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [10]:
# Adjust width/height as needed

Image(url="mentions-largest-cc.png", width=1200)

In this graph, we can observe the relationship between Twitter accounts. The node size and label size are proportional to their in-degree (the greater the in-degree, the more mentions on Twitter). The network diameter is 10, which can be found after analyzing the undirected version of the network. The largest connected component has 90 nodes, representing 25.21% of the entire graph (90/357 * 100). The second largest connected component consists of 17 nodes.


## 2.2. Cluster the largest connected component


Yes, most of the small nodes surrounding the @elnacionalcat node belong to the cluster. Some of the members include RoserPrG, Winona1610, and tsunamilluis. One reason these nodes may belong to the cluster is that they share similar interests or opinions regarding the COVID situation. 


## 2.3. Examine degree distributions


In [12]:
# Adjust width/height as needed

display(Image(url="mentions-largest-cc-indegree.png", width=500))

display(Image(url="mentions-largest-cc-outdegree.png", width=500))

We can observe that the in-degree distribution follows a normal pattern, with more nodes having fewer connections and fewer nodes having a larger degree. However, in the out-degree distribution, the graph is skewed to the right. This might be because the out-degree refers to the number of people you mention, and there are fewer people who don't mention anyone. The number of mentions the most nodes have corresponds to the peak in the graph.

# 3. Create the undirected co-mention network


In [13]:
co_mentions_counter = {}

with gzip.open(COMPRESSED_INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
    for line in input_file:
        tweet = json.loads(line)
        mentions = extract_mentions(tweet["full_text"])
        for mention1 in mentions:
            for mention2 in mentions:
                if mention1 < mention2:
                    key = (mention1, mention2)
                    if key in co_mentions_counter:
                        co_mentions_counter[key] += 1
                    else:
                        co_mentions_counter[key] = 1

In [14]:
# KEEP AS-IS

print(co_mentions_counter[('educaciocat', 'elnacionalcat')])

8


In [15]:
for (user1, user2), count in co_mentions_counter.items():
    if count >= 20:
        print(f"Accounts @{user1} and @{user2} have been co-mentioned {count} times")


Accounts @VilaWeb and @mossos have been co-mentioned 20 times
Accounts @QuimTorraiPla and @govern have been co-mentioned 92 times
Accounts @sanchezcastejon and @tjparfitt have been co-mentioned 28 times
Accounts @elnacionalcat and @joseantich have been co-mentioned 90 times
Accounts @QuimTorraiPla and @tjparfitt have been co-mentioned 59 times
Accounts @QuimTorraiPla and @sanchezcastejon have been co-mentioned 25 times
Accounts @emergenciescat and @govern have been co-mentioned 31 times
Accounts @josepcosta and @sanchezcastejon have been co-mentioned 49 times
Accounts @eldiarioes and @iescolar have been co-mentioned 28 times
Accounts @gencat and @govern have been co-mentioned 105 times
Accounts @mossos and @semgencat have been co-mentioned 44 times
Accounts @elnacionalcat and @juansrod1 have been co-mentioned 30 times
Accounts @QuimTorraiPla and @emergenciescat have been co-mentioned 75 times
Accounts @Antoni_Gelonch and @sanchezcastejon have been co-mentioned 106 times


In [16]:
with io.open(OUTPUT_CO_MENTIONS_FILENAME, "w") as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"')
    writer.writerow(["Source", "Target", "Weight"])
    for (user1, user2), weight in co_mentions_counter.items():
        writer.writerow([user1, user2, weight])

# 4. Visualize the undirected co-mention network in Cytoscape


In [17]:
# Adjust width/height as needed

Image(url="co-mentions-min-degree-17.png", width=1200)

The graph visualizes the undirected co-mention network of Twitter accounts. It contains several connected components, with the largest connected component being the most prominent. The connected components vary in density; some are sparse with few connections, while others are denser. Within the largest connected component, there is a particularly dense sub-graph where certain accounts are frequently co-mentioned together, indicating a strong relationship or common topic of discussion among those accounts. This dense sub-graph likely represents a cluster of accounts that are thematically related or frequently interact with each other. Overall, the graph highlights the varying levels of interaction and co-mentioning among different groups of Twitter accounts.